In [1]:
from multiprocessing import Pool
import subprocess
import yaml
import os
import sys
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
import tempfile
import pandas as pd
from collections import Counter
import numpy as np
import threading
source_path = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
sys.path.append(source_path)

from sklearn.metrics import accuracy_score
from joblib import load
import warnings
warnings.filterwarnings("ignore")

import torch
import torch.nn as nn
import torch.nn.functional as F

from PIL import Image, ImageDraw, ImageFont
import matplotlib.pyplot as plt
import cv2

this_path = os.path.abspath(os.getcwd())
source_path = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
sys.path.append(source_path)
dataset='iam' #'icdar'
save_debug_images=False

In [4]:
selected_FE = 'clip-vit-large-patch14-inter'#'clip-vit-large-patch14'#'resnet50'  # Example feature extractor, can be changed
selected_classification_head = 'MLPClassifier1'
models_dir = os.path.join(this_path, 'models', selected_FE)
train_filename = models_dir+f'/{selected_FE}_features_{dataset}_train_df_w_patches.csv' 
mean,scale = model_utils.get_normalization_parameters(train_filename)
model_parameters = {
    'n_neurons':256,
    'hidden_sizes': [128],  
    'dropout':0.6,
    'with_input_norm': 'batch_norm',  # Whether to use input normalization 'batch_norm' , None
    'mean': None,  # Mean for input normalization
    'scale': None,  # Scale for input normalization
    'activation': 'relu'
}
selected_metric = 'weighted_vote' #['majority_vote', 'weighted_vote', 'most_probable']

In [5]:
lang_train= '' #'ar' #if en only train on en, if ar only train on ar, if en+ar or '' train on both
lang_test = '' #'ar' #if en only test on en, if ar only test on ar, if en+ar or '' test on both
lang_sub = '_'+lang_train if lang_train in ['en', 'ar'] else ''
lang_test_sub = '_'+lang_test if lang_test in ['en', 'ar'] else ''

huggingface = global_vars.get_props(selected_FE).hugging
custom_pretrained = 'original'  # 'original', 'contrastive', 'fine-tune','progressive
evaluation_mode =  'full_model' #'backbone_only' #full_model

data_augmentation = False
suffix = '_augmented' if data_augmentation else ''
kind= 'patches_224'  # Example kind, can be changed

head_type = 'pytorch'  # 'sklearn' or 'pytorch'
if kind == 'body':
    selected_metric = 'most_probable'  # Use the metric you want to analyze

#prepare folders to save results
load_dir = os.path.join(models_dir,f'torch_model_trained_on_rep{lang_sub}',f'{selected_classification_head}', dataset)
search_dir=load_dir+'/current/'

explanation_dir, cam_dir, attention_dir, original_dir, transformed_dir, augmented_dir = file_IO.init_expl_dirs(search_dir,clear=True)

is_train = 'test' #val, test, train
if is_train == 'val':
    zarr=f'{dataset}_public_df_w_patches'
elif is_train == 'test':
    zarr=f'{dataset}_private_df_w_patches'
else:
    zarr=f'{dataset}_train_df_w_patches'
zarr_path=os.path.join(this_path, 'datasets', 'zarr', f'{zarr}.zarr')

In [6]:
classification_head, val_df = data_loading.load_classification_head(selected_FE,selected_classification_head,
                                                                    head_type,'val_only',train_filename, source_path,
                                                                    **model_parameters,custom_pretrained=custom_pretrained, 
                                                                    lang=lang_sub, search_dir=load_dir)
transform = u_transforms.get_transform(selected_FE, use_patches=True, custom=False, mode='resize')
backbone = model_utils.get_model(name=selected_FE, mode='truncated', pretrained=True, truncation='remove head')

model = model_utils.JoinedModels(backbone,classification_head)
selected=pd.read_csv(os.path.join(search_dir,f'selected_instances_{selected_metric}.csv'))

In [11]:
selected.columns

Index(['writer', 'isEng', 'same_text', 'file_name', 'male', 'Usage', 'x', 'y',
       'x2', 'y2', 'n_cc', 'black_ratio', 'index', 'page', 'train', 'y_prob',
       'y_pred', 'grouped_true', 'majority_vote', 'majority_vote_uncertainty',
       'weighted_vote', 'weighted_vote_uncertainty', 'most_probable',
       'most_probable_uncertainty', 'y_pred_writer', 'y_prob_writer',
       'majority_vote_selected', 'majority_vote_sure', 'majority_vote_unsure',
       'majority_vote_ok', 'weighted_vote_selected', 'weighted_vote_sure',
       'weighted_vote_unsure', 'weighted_vote_ok', 'most_probable_selected',
       'most_probable_sure', 'most_probable_unsure', 'most_probable_ok'],
      dtype='object')

In [ ]:
if save_debug_images:
    zarr_visualizer=visualization.ZarrVisualizer(selected, zarr_path,selected_metric, search_dir, transform=transform, huggingface=huggingface, use_augmentation=True)
    zarr_visualizer.save_images(mode='original')
    zarr_visualizer.save_images(mode='preprocessed')
    zarr_visualizer.save_images(mode='augmentation')

In [7]:
print(len(selected))
#get unique file_name values 
unique_file_names = selected['file_name'].unique()
print(f"Number of unique file names: {len(unique_file_names)}")
#select one id at random
random_file_name = np.random.choice(unique_file_names)
print(f"Selected file name: {random_file_name}")

183
Number of unique file names: 61
Selected file name: C:\Users\andre\PhD\Datasets\iam online\lineImages-all\lineImages\e09\e09-515\e09-515z-04.tif


In [ ]:
#save explanations
mode='last'
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
cam_visualizations=explanation.gradcam_on_df(selected,model,transform,huggingface,device,selected_FE,mode=mode)
visualization.display_vis_on_background(cam_visualizations,selected,selected_metric=selected_metric,blank_background=False,save_path=cam_dir)
#visualization.display_gradcam_vis(cam_visualizations,selected,selected_metric=selected_metric, save_path=cam_dir)

17 40


ValueError: Unsupported model name: clip-vit-large-patch14-inter

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
backbone = model_utils.get_model(name=selected_FE, mode='exp', pretrained=True, truncation='remove head')
attention_masks=explanation.attention_on_df(selected,backbone,transform,huggingface,device,model_name=selected_FE,
                                            discard_ratio=0.9, head_fusion='max')
#attention_masks=explanation.attention_on_df(selected[selected['file_name'] == random_file_name],backbone,transform,huggingface,device,
 #                                           model_name=selected_FE, discard_ratio=0.9, head_fusion='max')
visualization.display_vis_on_background(attention_masks,selected,mask=True,selected_metric=selected_metric,blank_background=False,save_path=attention_dir)

In [31]:
print(model)

JoinedModels(
  (vision_model): WrappedModelInter(
    (vision): CLIPVisionTransformer(
      (embeddings): CLIPVisionEmbeddings(
        (patch_embedding): Conv2d(3, 1024, kernel_size=(14, 14), stride=(14, 14), bias=False)
        (position_embedding): Embedding(257, 1024)
      )
      (pre_layrnorm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
      (encoder): CLIPEncoder(
        (layers): ModuleList(
          (0-23): 24 x CLIPEncoderLayer(
            (self_attn): CLIPAttention(
              (k_proj): Linear(in_features=1024, out_features=1024, bias=True)
              (v_proj): Linear(in_features=1024, out_features=1024, bias=True)
              (q_proj): Linear(in_features=1024, out_features=1024, bias=True)
              (out_proj): Linear(in_features=1024, out_features=1024, bias=True)
            )
            (layer_norm1): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
            (mlp): CLIPMLP(
              (activation_fn): QuickGELUActivation()
    

In [9]:
chefer_attention_dir = os.path.join(os.path.dirname(attention_dir), 'chefer_attention')
print(f"Chefer attention directory: {chefer_attention_dir}")
os.makedirs(chefer_attention_dir, exist_ok=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
attention_masks=explanation.attention_on_df(selected,model,transform,huggingface,device,model_name=selected_FE,
                                            discard_ratio=0.7, head_fusion='mean', method='chefer')
#attention_masks=explanation.attention_on_df(selected[selected['file_name'] == random_file_name],model,transform,huggingface,device,
 #                                           model_name=selected_FE, discard_ratio=0.9, head_fusion='max', method='chefer')
visualization.display_vis_on_background(attention_masks,selected,mask=True,selected_metric=selected_metric,blank_background=False,save_path=chefer_attention_dir)

Chefer attention directory: c:\Users\andre\VsCode\PD related projects\gender_detection\notebooks\pipeline_single_model\models\clip-vit-large-patch14-inter\torch_model_trained_on_rep\MLPClassifier1\iam/current/explanations\chefer_attention
633 341
706 343
690 395
457 317
679 438
509 315
723 396
630 360


In [33]:
mask = attention_masks[0][0]
m = mask.reshape(16, 16)
print("rows identical?", m.std(axis=0).mean())   # ~0 → uniform along y
print("cols identical?", m.std(axis=1).mean())
print("unique values:", len(np.unique(m)))       # ~16 → it's really 1-D
print(np.array2string(m, precision=2))

rows identical? 0.19114023
cols identical? 0.15669249
unique values: 256
[[1.   0.51 0.46 0.39 0.4  0.41 0.38 0.46 0.51 0.42 0.47 0.41 0.42 0.48
  0.48 0.7 ]
 [0.55 0.32 0.4  0.22 0.27 0.23 0.23 0.41 0.47 0.26 0.28 0.33 0.75 0.42
  0.42 0.55]
 [0.48 0.37 0.4  0.34 0.25 0.28 0.27 0.39 0.5  0.33 0.36 0.44 0.64 0.44
  0.62 0.91]
 [0.44 0.26 0.39 0.31 0.24 0.43 0.36 0.5  0.77 0.23 0.79 0.8  0.63 0.43
  0.78 0.96]
 [0.46 0.76 0.84 0.73 0.32 0.75 0.61 0.92 0.68 0.45 0.73 0.81 0.55 0.44
  0.76 0.62]
 [0.51 0.8  0.82 0.73 0.29 0.65 0.76 0.88 0.73 0.53 0.86 0.76 0.34 0.39
  0.74 0.89]
 [0.42 0.58 0.74 0.23 0.24 0.84 0.71 0.45 0.46 0.59 0.78 0.61 0.2  0.24
  0.23 0.43]
 [0.44 0.58 0.66 0.23 0.26 0.28 0.23 0.39 0.46 0.33 0.85 0.77 0.28 0.39
  0.3  0.39]
 [0.42 0.63 0.67 0.2  0.24 0.26 0.19 0.39 0.46 0.25 0.77 0.46 0.23 0.36
  0.25 0.43]
 [0.37 0.64 0.78 0.2  0.26 0.23 0.17 0.26 0.43 0.22 0.29 0.23 0.16 0.24
  0.22 0.32]
 [0.4  0.22 0.25 0.16 0.18 0.17 0.19 0.34 0.4  0.21 0.25 0.16 0.14 0.27
  0.2

In [20]:
#Dark background: cmap="turbo", gamma=0.7, alpha=0.6
#Light background: cmap="magma", gamma=1.0, alpha=0.4
#cmap: inferno, magma, plasma, viridis, cividis, coolwarm, turbo
enhanced_attention = visualization.enhance_attention(attention_masks, base_rgb=None, cmap_name="bwr",
                             pclip=(2,98), gamma=0.8, max_alpha=0.5,
                             draw_contours=False, contour_levels=(0.7, 0.9))

In [ ]:
#visualization.display_vis_on_background(attention_masks,selected,mask=True,selected_metric=selected_metric,blank_background=False,save_path=attention_dir)


224 224
224 224
224 224
224 224
224 224
224 224
224 224
224 224
224 224
224 224
224 224
224 224


# reload

In [2]:
def reload_modules():
    import importlib
    import utils.data_loading as data_loading
    import utils.visualization as visualization
    import utils.dataframes as dataframes
    import utils.utils_transforms as u_transforms
    import utils.training_utils as training_utils
    import utils.model_utils as model_utils
    import utils.file_IO as file_IO
    import utils.vit_rollout_mod as vit_rollout_mod
    import utils.explanation as explanation
    import utils.script_launching as script_launching
    import utils.global_vars as global_vars
    
    importlib.reload(file_IO)
    importlib.reload(data_loading)
    importlib.reload(visualization)
    importlib.reload(dataframes)
    importlib.reload(u_transforms)
    importlib.reload(model_utils)
    importlib.reload(training_utils)
    importlib.reload(vit_rollout_mod)
    importlib.reload(explanation)
    importlib.reload(script_launching)
    importlib.reload(global_vars)

    return data_loading, visualization, dataframes, u_transforms, training_utils, model_utils, file_IO, vit_rollout_mod, explanation, script_launching, global_vars
data_loading, visualization, dataframes, u_transforms, training_utils, model_utils, file_IO, vit_rollout_mod, explanation, script_launching, global_vars = reload_modules()